In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
ROOT = Path.cwd()

if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent

RAW = ROOT / "data" / "raw" / "kobo_export.csv"
PROCESSED = ROOT / "data" / "processed"

PROCESSED.mkdir(parents=True, exist_ok=True)

df_raw = pd.read_csv(RAW, sep=";")

print("Raw file exists:", RAW.exists())
print("Rows:", len(df_raw))
print("Columns:", len(df_raw.columns))
print("\nFirst 10 columns:")
print(df_raw.columns[:10].tolist())

Raw file exists: True
Rows: 15
Columns: 37

First 10 columns:
['start', 'end', 'Facility_name', 'Facility_level', 'County', 'Reporting_period', 'OPD_visits', 'ANC_visits', 'PNC_visits', 'Immunization_visits']


In [3]:
df = df_raw.copy()

# Standardize Kobo column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
    .str.replace("/", "_", regex=False)
)

# Rename project-specific fields
df = df.rename(columns={
    "referral": "referrals",
    "data_verification_completed": "verification_completed",
    "calculation": "total_service_contacts",
    "calculation_001": "referral_rate",
    "calculation_002": "stockout_flag"
})

print("facility_name exists:",
      "facility_name" in df.columns)

print("opd_visits exists:",
      "opd_visits" in df.columns)

print("referrals exists:",
      "referrals" in df.columns)

facility_name exists: True
opd_visits exists: True
referrals exists: True


In [4]:
register_map = {
    "which_registers_were_missing_opd_register": "OPD Register",
    "which_registers_were_missing_anc_register": "ANC Register",
    "which_registers_were_missing_pnc_register": "PNC Register",
    "which_registers_were_missing_immunization_register": "Immunization Register",
    "which_registers_were_missing_pharmacy_register": "Pharmacy Register"
}

def combine_registers(row):
    selected = []

    for column, label in register_map.items():
        if column in row.index:
            value = str(row[column]).strip().lower()

            if value not in {"", "nan", "no", "false", "0"}:
                selected.append(label)

    return ", ".join(selected)

df["missing_register_types"] = df.apply(
    combine_registers,
    axis=1
)

text_cols = [
    "facility_name",
    "facility_level",
    "county",
    "stockout_occurred",
    "product_affected",
    "reporting_completed",
    "missing_registers",
    "missing_register_types",
    "verification_completed"
]

for col in text_cols:
    df[col] = df[col].astype("string").str.strip()

df["reporting_period"] = pd.to_datetime(
    df["reporting_period"],
    errors="coerce"
)

numeric_cols = [
    "opd_visits",
    "anc_visits",
    "pnc_visits",
    "immunization_visits",
    "referrals",
    "stockout_days",
    "total_service_contacts",
    "referral_rate",
    "stockout_flag"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

print("Cleaning completed.")
print("Records:", len(df))

Cleaning completed.
Records: 15


In [5]:
checks = []

def add_check(name, condition):
    failed = int(condition.sum())

    checks.append({
        "check": name,
        "failed_records": failed,
        "status": "PASS" if failed == 0 else "FAIL"
    })

required = [
    "facility_name",
    "facility_level",
    "county",
    "reporting_period",
    "opd_visits",
    "anc_visits",
    "pnc_visits",
    "immunization_visits",
    "referrals"
]

add_check(
    "Required fields",
    df[required].isna().any(axis=1)
)

add_check(
    "OPD range",
    (df["opd_visits"] < 0) |
    (df["opd_visits"] > 10000)
)

add_check(
    "ANC range",
    (df["anc_visits"] < 0) |
    (df["anc_visits"] > 3000)
)

add_check(
    "PNC range",
    (df["pnc_visits"] < 0) |
    (df["pnc_visits"] > 3000)
)

add_check(
    "Immunization range",
    (df["immunization_visits"] < 0) |
    (df["immunization_visits"] > 5000)
)

add_check(
    "Referral range",
    (df["referrals"] < 0) |
    (df["referrals"] > 2000)
)

add_check(
    "Referrals <= OPD",
    df["referrals"] > df["opd_visits"]
)

add_check(
    "Stockout days valid",
    (
        (df["stockout_occurred"].str.lower() == "yes") &
        (
            df["stockout_days"].isna() |
            (df["stockout_days"] < 1) |
            (df["stockout_days"] > 31)
        )
    )
)

add_check(
    "Reporting date not future",
    df["reporting_period"] >
    pd.Timestamp.today().normalize()
)

# Independent calculation checks
expected_contacts = (
    df["opd_visits"]
    + df["anc_visits"]
    + df["pnc_visits"]
    + df["immunization_visits"]
)

expected_rate = np.where(
    df["opd_visits"] == 0,
    0,
    df["referrals"] / df["opd_visits"] * 100
)

expected_flag = np.where(
    df["stockout_occurred"].str.lower() == "yes",
    1,
    0
)

add_check(
    "Total contacts calculation",
    ~np.isclose(
        df["total_service_contacts"],
        expected_contacts,
        atol=0.01,
        equal_nan=True
    )
)

add_check(
    "Referral rate calculation",
    ~np.isclose(
        df["referral_rate"],
        expected_rate,
        atol=0.01,
        equal_nan=True
    )
)

add_check(
    "Stockout flag calculation",
    df["stockout_flag"] != expected_flag
)

validation = pd.DataFrame(checks)

display(validation)

overall = validation["status"].eq("PASS").all()

print("\nOverall validation:",
      "PASS" if overall else "FAIL")

,check,failed_records,status
0,Required fields,0,PASS
1,OPD range,0,PASS
2,ANC range,0,PASS
3,PNC range,0,PASS
4,Immunization range,0,PASS
5,Referral range,0,PASS
6,Referrals <= OPD,0,PASS
7,Stockout days valid,0,PASS
8,Reporting date not future,0,PASS
9,Total contacts calculation,0,PASS



Overall validation: PASS


In [6]:
clean_file = PROCESSED / "kobo_clean.csv"
validation_file = PROCESSED / "kobo_validation_results.csv"

df.to_csv(
    clean_file,
    index=False
)

validation.to_csv(
    validation_file,
    index=False
)

print("PYTHON ETL COMPLETE")
print("-" * 35)
print("Raw records:", len(df_raw))
print("Clean records:", len(df))
print("Validation checks:", len(validation))
print("All checks passed:", overall)
print("\nClean file:", clean_file)
print("Validation file:", validation_file)

PYTHON ETL COMPLETE
-----------------------------------
Raw records: 15
Clean records: 15
Validation checks: 12
All checks passed: True

Clean file: c:\Users\USER\Desktop\kobo-facility-service-mne\data\processed\kobo_clean.csv
Validation file: c:\Users\USER\Desktop\kobo-facility-service-mne\data\processed\kobo_validation_results.csv


In [7]:
# Create a database-ready CSV for MySQL import

mysql_columns = [
    "facility_name",
    "facility_level",
    "county",
    "reporting_period",
    "opd_visits",
    "anc_visits",
    "pnc_visits",
    "immunization_visits",
    "referrals",
    "stockout_occurred",
    "product_affected",
    "stockout_days",
    "reporting_completed",
    "missing_registers",
    "missing_register_types",
    "verification_completed",
    "total_service_contacts",
    "referral_rate",
    "stockout_flag"
]

df_mysql = df[mysql_columns].copy()

mysql_file = PROCESSED / "kobo_mysql_import.csv"

df_mysql.to_csv(
    mysql_file,
    index=False
)

print("MySQL import file created:")
print(mysql_file)

print("\nRows:", len(df_mysql))
print("Columns:", len(df_mysql.columns))

display(df_mysql.head())

MySQL import file created:
c:\Users\USER\Desktop\kobo-facility-service-mne\data\processed\kobo_mysql_import.csv

Rows: 15
Columns: 19


,facility_name,facility_level,county,reporting_period,opd_visits,anc_visits,pnc_visits,immunization_visits,referrals,stockout_occurred,product_affected,stockout_days,reporting_completed,missing_registers,missing_register_types,verification_completed,total_service_contacts,referral_rate,stockout_flag
0,Coast General Test Facility,level_5,mombasa,2026-01-31,1800,320,210,650,90,no,<NA>,NaN,yes,no,,yes,2980,5.000000,0
1,Likoni Test Health Center,level_3,mombasa,2026-01-31,850,180,120,300,45,yes,amoxicillin,5.0,yes,no,,yes,1450,5.294118,1
2,Kilifi Central Test Facility,level_4,kilifi,2026-02-28,1400,260,175,520,80,no,<NA>,NaN,yes,no,,yes,2355,5.714286,0
3,Malindi Test Hospital,level_4,kilifi,2026-02-28,2200,410,280,700,140,yes,paracetamol,8.0,yes,yes,"OPD Register, ANC Register, PNC Register, Immu...",no,3590,6.363636,1
4,Kwale Subcounty Test Facility,level_3,kwale,2026-03-31,950,210,145,380,55,no,<NA>,NaN,yes,no,,yes,1685,5.789474,0


In [8]:
mysql_columns = [
    "facility_name",
    "facility_level",
    "county",
    "reporting_period",
    "opd_visits",
    "anc_visits",
    "pnc_visits",
    "immunization_visits",
    "referrals",
    "stockout_occurred",
    "product_affected",
    "stockout_days",
    "reporting_completed",
    "missing_registers",
    "missing_register_types",
    "verification_completed",
    "total_service_contacts",
    "referral_rate",
    "stockout_flag"
]

mysql_import = df[mysql_columns].copy()

mysql_file = PROCESSED / "kobo_mysql_import.csv"

mysql_import.to_csv(
    mysql_file,
    index=False
)

print("MySQL import file created.")
print("Rows:", len(mysql_import))
print("Columns:", len(mysql_import.columns))
print("File:", mysql_file)

display(mysql_import)

MySQL import file created.
Rows: 15
Columns: 19
File: c:\Users\USER\Desktop\kobo-facility-service-mne\data\processed\kobo_mysql_import.csv


,facility_name,facility_level,county,reporting_period,opd_visits,anc_visits,pnc_visits,immunization_visits,referrals,stockout_occurred,product_affected,stockout_days,reporting_completed,missing_registers,missing_register_types,verification_completed,total_service_contacts,referral_rate,stockout_flag
0,Coast General Test Facility,level_5,mombasa,2026-01-31,1800,320,210,650,90,no,<NA>,NaN,yes,no,,yes,2980,5.000000,0
1,Likoni Test Health Center,level_3,mombasa,2026-01-31,850,180,120,300,45,yes,amoxicillin,5.0,yes,no,,yes,1450,5.294118,1
2,Kilifi Central Test Facility,level_4,kilifi,2026-02-28,1400,260,175,520,80,no,<NA>,NaN,yes,no,,yes,2355,5.714286,0
3,Malindi Test Hospital,level_4,kilifi,2026-02-28,2200,410,280,700,140,yes,paracetamol,8.0,yes,yes,"OPD Register, ANC Register, PNC Register, Immu...",no,3590,6.363636,1
4,Kwale Subcounty Test Facility,level_3,kwale,2026-03-31,950,210,145,380,55,no,<NA>,NaN,yes,no,,yes,1685,5.789474,0
5,Msambweni Test Facility,level_4,kwale,2026-03-31,1250,300,190,450,100,yes,artemether_lumefantrine,12.0,yes,no,,yes,2190,8.000000,1
6,Taita Test Hospital,level_4,taita_taveta,2026-04-30,1100,230,150,400,65,no,<NA>,NaN,yes,no,,yes,1880,5.909091,0
7,Voi Test Health Center,level_3,taita_taveta,2026-04-30,720,160,105,280,40,yes,ors,3.0,yes,no,,yes,1265,5.555556,1
8,Nakuru Test Hospital,level_5,nakuru,2026-05-31,2600,500,350,850,180,no,<NA>,NaN,yes,no,,yes,4300,6.923077,0
9,Naivasha Test Facility,level_4,nakuru,2026-05-31,1750,340,220,600,125,yes,zinc,7.0,yes,no,,yes,2910,7.142857,1


In [9]:
check = pd.read_csv(mysql_file)

print("CSV rows:", len(check))
print("CSV columns:", len(check.columns))
print("First index:", check.index.min())
print("Last index:", check.index.max())

CSV rows: 15
CSV columns: 19
First index: 0
Last index: 14
